# 규칙 기반 훈련 맥락 요약

3장에서 만든 날짜별 추천 입력 데이터를 불러와, 추천일 이전 기록만으로 훈련 맥락을 한 줄로 요약하는 규칙을 만든다.

이 단계의 규칙은 실제 피로·회복 상태나 특정 훈련 처방을 확정하지 않는다.

In [30]:
from pathlib import Path

import pandas as pd


project_root = Path("..").resolve()
recommendation_output_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_recommendation_context.csv"
)

recommendation_output_df = pd.read_csv(
    recommendation_output_path,
    parse_dates=["recommendation_date"],
)

recommendation_output_df.shape

(100, 13)

In [31]:
recommendation_output_df = (
    recommendation_output_df
    .set_index("recommendation_date")
)

recommendation_date = "2009-06-02"

recommendation_output_df.loc[
    [recommendation_date]
].round(2)

,previous_7_day_tss,previous_28_day_tss,short_to_long_tss_ratio,previous_ratio_q25,previous_ratio_q75,relative_recent_load_level,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days,previous_day_record_status,recommendation_context_type
recommendation_date,,,,,,,,,,,,
2009-06-02 00:00:00+00:00,895.56,2949.45,1.21,0.88,1.15,relatively_high,13.33,8.0,12.0,0.0,recorded,relatively_high_after_recorded


In [32]:
def summarize_recommendation_context(row):
    load_messages = {
        "relatively_high": "최근 부하가 과거 흐름보다 높은 편입니다.",
        "typical_range": "최근 부하는 과거 흐름에서 일반적인 범위입니다.",
        "relatively_low": "최근 부하가 과거 흐름보다 낮은 편입니다.",
    }

    if row["previous_day_record_status"] == "recorded":
        previous_day_message = (
            "전날에 라이드 기록이 있으며, "
            f"연속 라이드 기록은 "
            f"{int(row['previous_day_recorded_ride_streak_days'])}일입니다."
        )
    else:
        previous_day_message = (
            "전날에는 라이드 기록이 없으며, "
            f"연속 무기록은 "
            f"{int(row['previous_day_no_record_streak_days'])}일입니다."
        )

    return " ".join(
        [
            load_messages[row["relative_recent_load_level"]],
            (
                f"최근 7일 TSS 비율은 "
                f"{row['short_to_long_tss_ratio']:.2f}입니다."
            ),
            previous_day_message,
        ]
    )

recommendation_example = recommendation_output_df.loc[recommendation_date]

summarize_recommendation_context(recommendation_example)

'최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.21입니다. 전날에 라이드 기록이 있으며, 연속 라이드 기록은 12일입니다.'

In [33]:
recommendation_output_df["recommendation_context_summary"] = (
    recommendation_output_df.apply(
        summarize_recommendation_context,
        axis=1,
    )
)

recommendation_output_df[
    [
        "recommendation_context_type",
        "recommendation_context_summary",
    ]
].head()

,recommendation_context_type,recommendation_context_summary
recommendation_date,,
2009-03-21 00:00:00+00:00,relatively_high_after_recorded,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.32입니다....
2009-03-22 00:00:00+00:00,relatively_high_after_recorded,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.39입니다....
2009-03-23 00:00:00+00:00,relatively_high_after_recorded,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.34입니다....
2009-03-24 00:00:00+00:00,relatively_high_after_no_record,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.17입니다....
2009-03-25 00:00:00+00:00,typical_range_after_no_record,최근 부하는 과거 흐름에서 일반적인 범위입니다. 최근 7일 TSS 비율은 0.96입...


In [34]:
representative_summaries = (
    recommendation_output_df
    .groupby("recommendation_context_type")
    .head(1)
    [
        [
            "recommendation_context_type",
            "recommendation_context_summary",
        ]
    ]
)

representative_summaries.to_string()

'                               recommendation_context_type                                                        recommendation_context_summary\nrecommendation_date                                                                                                                             \n2009-03-21 00:00:00+00:00   relatively_high_after_recorded     최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.32입니다. 전날에 라이드 기록이 있으며, 연속 라이드 기록은 1일입니다.\n2009-03-24 00:00:00+00:00  relatively_high_after_no_record       최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.17입니다. 전날에는 라이드 기록이 없으며, 연속 무기록은 1일입니다.\n2009-03-25 00:00:00+00:00    typical_range_after_no_record    최근 부하는 과거 흐름에서 일반적인 범위입니다. 최근 7일 TSS 비율은 0.96입니다. 전날에는 라이드 기록이 없으며, 연속 무기록은 2일입니다.\n2009-03-28 00:00:00+00:00     typical_range_after_recorded  최근 부하는 과거 흐름에서 일반적인 범위입니다. 최근 7일 TSS 비율은 0.78입니다. 전날에 라이드 기록이 있으며, 연속 라이드 기록은 1일입니다.\n2009-03-29 00:00:00+00:00    relatively_low_after_recorded     최근 부하가 과거 흐름보다 낮은 편입니다. 최근 7일 TSS 비율은 0.68입니다. 전날에 라이드 기록이 있

In [35]:
recommendation_messages = {
    "relatively_high_after_recorded": (
        "최근 부하와 전날 기록이 이어져 있어, "
        "다음 훈련 강도는 보수적으로 검토합니다."
    ),
    "relatively_high_after_no_record": (
        "최근 부하는 높은 편입니다. 전날 무기록만으로 "
        "휴식을 단정하지 않고, 다음 훈련 강도는 보수적으로 검토합니다."
    ),
    "typical_range_after_recorded": (
        "최근 부하는 일반 범위이며 전날 기록이 있어, "
        "계획된 훈련을 검토합니다."
    ),
    "typical_range_after_no_record": (
        "최근 부하는 일반 범위입니다. 전날 무기록 여부를 함께 확인한 뒤, "
        "계획된 훈련을 검토합니다."
    ),
    "relatively_low_after_recorded": (
        "최근 부하는 낮은 편이지만 전날 기록이 있어, "
        "점진적인 부하 조절 가능성을 검토합니다."
    ),
    "relatively_low_after_no_record": (
        "최근 부하가 낮고 전날 기록도 없습니다. 기록 맥락을 확인한 뒤, "
        "점진적인 훈련 재개 가능성을 검토합니다."
    ),
}

recommendation_output_df["recommendation_guidance"] = (
    recommendation_output_df["recommendation_context_type"]
    .map(recommendation_messages)
)

recommendation_output_df[
    [
        "recommendation_context_summary",
        "recommendation_guidance",
    ]
].head()

,recommendation_context_summary,recommendation_guidance
recommendation_date,,
2009-03-21 00:00:00+00:00,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.32입니다....,"최근 부하와 전날 기록이 이어져 있어, 다음 훈련 강도는 보수적으로 검토합니다."
2009-03-22 00:00:00+00:00,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.39입니다....,"최근 부하와 전날 기록이 이어져 있어, 다음 훈련 강도는 보수적으로 검토합니다."
2009-03-23 00:00:00+00:00,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.34입니다....,"최근 부하와 전날 기록이 이어져 있어, 다음 훈련 강도는 보수적으로 검토합니다."
2009-03-24 00:00:00+00:00,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.17입니다....,"최근 부하는 높은 편입니다. 전날 무기록만으로 휴식을 단정하지 않고, 다음 훈련 강..."
2009-03-25 00:00:00+00:00,최근 부하는 과거 흐름에서 일반적인 범위입니다. 최근 7일 TSS 비율은 0.96입...,"최근 부하는 일반 범위입니다. 전날 무기록 여부를 함께 확인한 뒤, 계획된 훈련을 ..."


In [36]:
recommendation_output_df[
    "recommendation_guidance"
].isna().sum()

np.int64(0)

In [37]:
recommendation_output_df.groupby(
    [
        "recommendation_context_type",
        "recommendation_guidance",
    ]
).size()

recommendation_context_type      recommendation_guidance                                      
relatively_high_after_no_record  최근 부하는 높은 편입니다. 전날 무기록만으로 휴식을 단정하지 않고, 다음 훈련 강도는 보수적으로 검토합니다.     4
relatively_high_after_recorded   최근 부하와 전날 기록이 이어져 있어, 다음 훈련 강도는 보수적으로 검토합니다.                     19
relatively_low_after_no_record   최근 부하가 낮고 전날 기록도 없습니다. 기록 맥락을 확인한 뒤, 점진적인 훈련 재개 가능성을 검토합니다.       9
relatively_low_after_recorded    최근 부하는 낮은 편이지만 전날 기록이 있어, 점진적인 부하 조절 가능성을 검토합니다.                 10
typical_range_after_no_record    최근 부하는 일반 범위입니다. 전날 무기록 여부를 함께 확인한 뒤, 계획된 훈련을 검토합니다.             20
typical_range_after_recorded     최근 부하는 일반 범위이며 전날 기록이 있어, 계획된 훈련을 검토합니다.                         38
dtype: int64

In [38]:
rule_based_output_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_rule_based_recommendations.csv"
)

rule_based_output_df = (
    recommendation_output_df
    .reset_index()
)

rule_based_output_df.to_csv(
    rule_based_output_path,
    index=False,
)

rule_based_output_df.shape

(100, 15)

### 규칙 기반 추천 문장 생성 결과

- 3장에서 저장한 100개 추천일의 훈련 맥락을 불러와, 최근 부하 수준과 전날 기록 상태를 한 줄 요약으로 만들었다.
- 최근 부하 수준과 전날 기록·무기록 상태를 조합한 여섯 가지 맥락 유형마다 검토용 추천 문장을 연결했다.
- 적용일 수는 일반 범위·전날 기록 38일, 일반 범위·전날 무기록 20일, 높은 부하·전날 기록 19일, 낮은 부하·전날 기록 10일, 낮은 부하·전날 무기록 9일, 높은 부하·전날 무기록 4일이다.
- 100개 추천일 모두에 추천 문장이 생성되었으며, 전날 무기록은 휴식으로 단정하지 않고 기록이 없다는 맥락으로만 표현했다.
- 날짜, 추천 입력값, 맥락 요약, 추천 문장을 포함한 100행 15열 결과를 `data/processed/goldencheetah_rule_based_recommendations.csv`로 저장했다.
- 이 규칙은 추천일 이전 기록만 사용한 검토용 안내이며, 실제 피로·회복 상태나 개별 훈련 처방을 확정하지 않는다.